## Import packages

In [ ]:
# Importing packages
import os
from pathlib import Path

import pandas as pd
import numpy as np

import scanpy as sc
import squidpy as sq
import anndata as ad
import rapids_singlecell as rs

import json

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

### Preventing OOM error for VRAM

In [ ]:
import rmm
import cupy as cp
from rmm.allocators.cupy import rmm_cupy_allocator

rmm.reinitialize(managed_memory=True, pool_allocator=False)
cp.cuda.set_allocator(rmm_cupy_allocator)

In [ ]:
# Set working directory
project = Path("/media/nannu1375/Backpack/Shankara/KNC")

os.chdir(project)

## Sample Loading

In [ ]:
# Read the h5 files
adata = sc.read_10x_h5(
    filename=project /
    "public_data/Spatial_data/Output-suppli_files/binned_outputs/square_008um/filtered_feature_bc_matrix.h5"
    )

In [ ]:
# Read the tissue positions file
positions = pd.read_parquet(
    project /
    "public_data/Spatial_data/Output-suppli_files/binned_outputs/square_008um/spatial/tissue_positions.parquet"
)

In [ ]:
# Join the positions file with the anndata file
## We are going to merge the positions file with obs first, then take the x y coordinates and put it in obsm since it just needs a matrix of coordinates.
adata.obs = adata.obs.join(
    positions.set_index("barcode")
)

# create the obsm x-y coordinates
adata.obsm["spatial"] = adata.obs[
    ["pxl_col_in_fullres", "pxl_row_in_fullres"]
].to_numpy()

In [ ]:
# read the scale factors
with open(
    project /
    "public_data/Spatial_data/Output-suppli_files/binned_outputs/square_008um/spatial/scalefactors_json.json"
) as f:
    scalefactors = json.load(f)

# Read the h&e images
hires = mpimg.imread(
    project /
    "public_data/Spatial_data/Output-suppli_files/binned_outputs/square_008um/spatial/tissue_hires_image.png"
)

lowres = mpimg.imread(
    project /
    "public_data/Spatial_data/Output-suppli_files/binned_outputs/square_008um/spatial/tissue_lowres_image.png"
)

# Display the image
plt.imshow(hires)
scalefactors

In [ ]:
# Build the uns dictionary
adata.uns["spatial"] = {
    "Visium_HD": {
        "images":{
            "hires": hires,
            "lowres": lowres,
        },
        "scalefactors": scalefactors,
        "metadata": {}
    }
}

## Sanity checks

In [ ]:
# =============================================================================
# Sanity checks for the Spatial AnnData object
# =============================================================================

print("=" * 70)
print("AnnData Summary")
print("=" * 70)
print(adata)

print("\n")

# =============================================================================
# Check dimensions
# =============================================================================

print("=" * 70)
print("Dimensions")
print("=" * 70)

print(f"Number of spatial bins (obs): {adata.n_obs:,}")
print(f"Number of genes (var):        {adata.n_vars:,}")

print("\n")

# =============================================================================
# Check available AnnData slots
# =============================================================================

print("=" * 70)
print("Available Slots")
print("=" * 70)

print("obsm keys :", list(adata.obsm.keys()))
print("uns keys  :", list(adata.uns.keys()))

print("\n")

# =============================================================================
# Check observation (spot/bin) metadata
# =============================================================================

print("=" * 70)
print("Observation metadata (adata.obs)")
print("=" * 70)

display(adata.obs.head())

print(f"\nShape : {adata.obs.shape}")

print("\n")

# =============================================================================
# Check gene metadata
# =============================================================================

print("=" * 70)
print("Gene metadata (adata.var)")
print("=" * 70)

display(adata.var.head())

print(f"\nShape : {adata.var.shape}")

print("\n")

# =============================================================================
# Check expression matrix
# =============================================================================

print("=" * 70)
print("Expression Matrix (adata.X)")
print("=" * 70)

print(type(adata.X))
print("Shape :", adata.X.shape)

print("\n")

# =============================================================================
# Check spatial coordinates
# =============================================================================

print("=" * 70)
print("Spatial Coordinates")
print("=" * 70)

print(type(adata.obsm["spatial"]))
print("Shape :", adata.obsm["spatial"].shape)

print("\nFirst five coordinates:")
print(adata.obsm["spatial"][:5])

print("\n")

# =============================================================================
# Check spatial metadata
# =============================================================================

print("=" * 70)
print("Spatial Metadata")
print("=" * 70)

library_id = list(adata.uns["spatial"].keys())[0]

print("Library ID :", library_id)

print("\nAvailable entries:")
print(list(adata.uns["spatial"][library_id].keys()))

print("\nScale factors:")
print(adata.uns["spatial"][library_id]["scalefactors"])

print("\n")

# =============================================================================
# Verify consistency
# =============================================================================

print("=" * 70)
print("Consistency Checks")
print("=" * 70)

assert adata.n_obs == adata.obs.shape[0], "Mismatch: obs rows"

assert adata.n_obs == adata.obsm["spatial"].shape[0], \
    "Mismatch: spatial coordinates"

assert adata.n_vars == adata.var.shape[0], \
    "Mismatch: gene metadata"

print("✓ Number of observations matches obs")
print("✓ Number of observations matches spatial coordinates")
print("✓ Number of genes matches var")

print("\nAll sanity checks passed!")

In [ ]:
# Create a copy of the raw counts
adata.layers["counts"] = adata.X.copy()

print(adata.layers["counts"])

## QC step

In [ ]:
# Annotate mitochondrial genes
adata.var["mt"] = adata.var_names.str.startswith("MT-")

# Display the mitochondrial genes and how many
print("Number of mitochondrial and nuclear genes (False is nuclear and true is mitochondrial): ", adata.var["mt"].value_counts())
print("List of gene names of the mitochondrial genes:\n", adata.var_names[adata.var["mt"] == True])

In [ ]:
# Calculating QC metrics
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt"],
    inplace=True
)

In [ ]:
print(adata.obs.columns)
print(adata.obs.head())

In [ ]:
# Get summary of qc metrics
print(adata.obs[
    ["total_counts",
    "n_genes_by_counts",
    "pct_counts_mt"]
].describe())

print(adata.obs[
    ["total_counts",
     "n_genes_by_counts",
     "pct_counts_mt"]
].quantile(
    [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
))

In [ ]:
# Rename duplicate genes
dedup = adata.var_names_make_unique()
if dedup == True:
    print("Duplicate gene names removed!")

In [ ]:
# Plot the qc metrics summary
adata.obs["total_counts"].hist(bins=100)
plt.xlim(0, 1500)
plt.title("Distribution of Total UMI Counts per Spatial Bin")
plt.xlabel("Total UMI counts")
plt.ylabel("Number of spatial bins")

plt.savefig(
    "Plots/Spatial/QC/umi_counts_hist.png",
    dpi=600,
    bbox_inches="tight"
)
plt.show()


In [ ]:
adata.obs["n_genes_by_counts"].hist(bins=100)
plt.xlim(0, 1100)
plt.title("Distribution of Detected Genes per Spatial Bin")
plt.xlabel("Number of detected genes")
plt.ylabel("Number of spatial bins")

plt.savefig(
    "Plots/Spatial/QC/genes_counts_hist.png",
    dpi=600,
    bbox_inches="tight"
)
plt.show()

In [ ]:
adata.obs["pct_counts_mt"].hist(bins=100)
plt.xlim(0, 10)
plt.title("Distribution of Mitochondrial Content")
plt.xlabel("Mitochondrial counts (%)")
plt.ylabel("Number of spatial bins")

plt.savefig(
    "Plots/Spatial/QC/mtGenes_counts_hist.png",
    dpi=600,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# UMI vs gene count scatter plot
sc.pl.scatter(
    adata,
    x="total_counts",
    y="n_genes_by_counts",
    show=False 
)

plt.xlim(0, 2500)

plt.savefig(
    "Plots/Spatial/QC/umiVSgene.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close()

# mt vs gene count scatter plot
sc.pl.scatter(
    adata,
    x="total_counts",
    y="pct_counts_mt",
    show=False 
)

plt.xlim(0, 2500)

plt.savefig(
    "Plots/Spatial/QC/mtVSgene.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close()

In [ ]:
# Check how many genes would be lost if genes filtered 
for cutoff in [5, 10, 20, 30, 50, 75, 100]:
    removed = (adata.obs["n_genes_by_counts"] < cutoff).sum()
    pct = removed / adata.n_obs * 100
    print(f"< {cutoff:3} genes : {removed:7,d} bins ({pct:.2f}%)")

# Check how many bins would be lost if umi filtered 
for cutoff in [5, 10, 20, 30, 50, 75, 100]:
    removed = (adata.obs["total_counts"] < cutoff).sum()
    pct = removed / adata.n_obs * 100
    print(f"< {cutoff:3} UMIs : {removed:7,d} bins ({pct:.2f}%)")
    
# Check how many bins would be lost if umi filtered 
for cutoff in [0.5, 1, 2, 3, 4, 5]:
    removed = (adata.obs["pct_counts_mt"] < cutoff).sum()
    pct = removed / adata.n_obs * 100
    print(f"< {cutoff:3} mt genes : {removed:7,d} bins ({pct:.2f}%)")

In [ ]:
# Plot the umi count on tissue image
ax = sc.pl.embedding(
    adata,
    basis="spatial",
    color="total_counts",
    size=1,
    vmax="p99",
    show=False
)

fig = ax.figure

fig.savefig(
    "Plots/Spatial/QC/spatial_umi_counts.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Plot the gene count on tissue image
ax = sc.pl.embedding(
    adata,
    basis="spatial",
    color="n_genes_by_counts",
    size=1,
    vmax="p99",
    show=False 
)

fig = ax.figure

fig.savefig(
    "Plots/Spatial/QC/spatial_gene_counts.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()


In [ ]:
# Plot the mt count on tissue image
ax = sc.pl.embedding(
    adata,
    basis="spatial",
    color="pct_counts_mt",
    size=1,
    vmax="p99",
    show=False
)

fig = ax.figure

fig.savefig(
    "Plots/Spatial/QC/spatial_mt_counts.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Llets filter for umi and genes. Keeping cutoff of 20 for both
sc.pp.filter_cells(adata, min_counts=20)
sc.pp.filter_cells(adata, min_genes=20)

# Filter the genes which are expressed at least in 3 bins
sc.pp.filter_genes(adata, min_cells=3)

print(adata.shape)

adata.obs[
    ["total_counts", "n_genes_by_counts", "pct_counts_mt"]
].describe()

In [ ]:
# Check distribution of bins with high mt
adata.obs["pct_counts_mt"].describe(
    percentiles=[0.90,0.95,0.97,0.98,0.99,0.995]
)

### Spatial look of umi, gene and mt after filtering

In [ ]:
# Plot the umi count on tissue image
ax = sc.pl.embedding(
    adata,
    basis="spatial",
    color="total_counts",
    size=1,
    vmax="p99",
    show=False
)

fig = ax.figure

fig.savefig(
    "Plots/Spatial/QC/spatial_filtered_umi_counts.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Plot the gene count on tissue image
ax = sc.pl.embedding(
    adata,
    basis="spatial",
    color="n_genes_by_counts",
    size=1,
    vmax="p99",
    show=False 
)

fig = ax.figure

fig.savefig(
    "Plots/Spatial/QC/spatial_filtered_gene_counts.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()


In [ ]:
# Plot the mt count on tissue image
ax = sc.pl.embedding(
    adata,
    basis="spatial",
    color="pct_counts_mt",
    size=1,
    vmax="p99",
    show=False
)

fig = ax.figure

fig.savefig(
    "Plots/Spatial/QC/spatial_filtered_mt_counts.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

## Normalisation and log transformation

In [ ]:
# Normalise the reads 
sc.pp.normalize_total(
    adata,
    target_sum=1e4
)

print(adata.X.min())
print(adata.X.max())


# Log transform the reads
sc.pp.log1p(adata)

print(adata.X.min())
print(adata.X.max())

## Filter highly variable genes and plot PCA

In [ ]:
## Calculate highly variable genes
sc.pp.highly_variable_genes(
    adata,
    flavor="seurat",
    n_top_genes=3000
)

sc.pl.highly_variable_genes(adata)

In [ ]:
adata.var

In [ ]:
## Plot PCA
# Keep only HVGs
adata = adata[:, adata.var["highly_variable"]].copy()

# Calculate PCA
sc.tl.pca(
    adata,
    zero_center=False
)

# Plot PCA
sc.settings.figdir = "Plots/Spatial/Dimentionality"
sc.pl.pca_variance_ratio(
    adata,
    log=True,
    show=True,
    save="_log.png"
)

sc.pl.pca_variance_ratio(
    adata,
    show=True,
    log=False,
    save="_linear.png"
)

sc.pl.pca(
    adata,
    color="total_counts",
    vmax="p99",
    show=True,
    save="_total_counts.png"
)

sc.pl.pca(
    adata,
    color="pct_counts_mt",
    vmax="p99",
    show=True,
    save="_mt_pct.png"
)

sc.pl.pca(
    adata,
    color="n_genes_by_counts",
    vmax="p99",
    show=True,
    save="_gene_counts.png"
)

## k-nearest neighbour

In [ ]:
rs.pp.neighbors(
    adata,
    n_neighbors=15,
    n_pcs=30
)

## UMAP

In [ ]:
sc.tl.umap(
    adata,
    random_state=0
)

sc.pl.umap(
    adata,
    size=3,
    show=True,
    save=".png"
)